In [ ]:
# caypollard — Qwen2.5-VL-3B lit le même graphe de scène qu'opus-scene (banc relations, 440 images)
# Exécution : Exécution > Tout exécuter, avec un GPU T4 (Exécution > Modifier le type d'exécution).
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess, sys, tarfile
D = '/content/drive/MyDrive/caypollard/colab'
W = '/content/cay'
os.makedirs(W + '/data/derived', exist_ok=True)
tarfile.open(D + '/code.tar').extractall(W)
tarfile.open(D + '/relations.tar.gz').extractall(W + '/data/derived')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==5.17.0'], check=True)
print(len(os.listdir(W + '/data/derived/relations/images')), 'images')

In [ ]:
import glob, json, time, torch
sys.path.insert(0, W + '/src')
from caypollard.atelier.bricks import SCENE_POSTURES, SCENE_PROMPT, SCENE_RELATIONS, SCENE_ROLES
from PIL import Image
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
prompt = SCENE_PROMPT.split('\n', 2)[2].format(path='', roles=', '.join(SCENE_ROLES),
    postures=', '.join(SCENE_POSTURES), relations=', '.join(SCENE_RELATIONS))
print(torch.__version__, torch.cuda.get_device_name(0))
repo = 'Qwen/Qwen2.5-VL-3B-Instruct'
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(repo, torch_dtype=torch.float16, device_map='cuda').eval()
processor = AutoProcessor.from_pretrained(repo, min_pixels=256 * 28 * 28, max_pixels=640 * 28 * 28)

In [ ]:
# Chaque graphe est écrit tout de suite sur le Drive : une coupure ne perd rien, une relance reprend.
out = D + '/scene--qwen25vl3b-scene'
os.makedirs(out, exist_ok=True)
for path in sorted(glob.glob(W + '/data/derived/relations/images/*.jpg')):
    item = os.path.basename(path)[:-4]
    if os.path.exists(f'{out}/{item}.json'):
        continue
    start = time.time()
    messages = [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': prompt}]}]
    text = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=[text], images=[Image.open(path).convert('RGB')], return_tensors='pt').to('cuda')
    with torch.no_grad():
        ids = model.generate(**inputs, max_new_tokens=700, do_sample=False)
    answer = processor.batch_decode(ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]
    s = answer.find('{')
    try:
        data, _ = json.JSONDecoder().raw_decode(answer[s:]) if s >= 0 else ({}, 0)
    except json.JSONDecodeError:
        data = {}
    if not isinstance(data, dict):
        data = {}
    data = {'figures': [], 'relations': [], 'subject': '', **data, '_raw': answer,
            'seconds': round(time.time() - start, 1)}
    json.dump(data, open(f'{out}/{item}.json', 'w'), ensure_ascii=False)
    print(item, len(data.get('relations', [])), data['seconds'], flush=True)
open(D + '/FINI.txt', 'w').write(str(len(os.listdir(out))))
print('fini', len(os.listdir(out)))